# 🍷 Tech Challenge — Classificação de Qualidade de Vinhos
### FIAP | Pós-Graduação em Data Analytics

---

**Objetivo:** Desenvolver um modelo de Machine Learning capaz de prever se um vinho é de **alta qualidade (nota ≥ 7)** ou **baixa/média qualidade (nota < 7)**, com base em suas características físico-químicas.

**Pipeline:**
1. Compreensão do Problema
2. Análise Exploratória de Dados (EDA)
3. Pré-processamento
4. Desenvolvimento de Modelos
5. Avaliação e Comparação
6. Interpretação dos Resultados

## ⚙️ Instalação de Dependências

Execute esta célula apenas se estiver no **Google Colab**. As bibliotecas principais (`pandas`, `scikit-learn`, `matplotlib`, `seaborn`) já vêm pré-instaladas no Colab.

In [ ]:
# Instala bibliotecas adicionais caso necessário
# No Google Colab, a maioria já está disponível por padrão
!pip install -q imbalanced-learn

## 📦 Importação das Bibliotecas

In [ ]:
# ── Manipulação de Dados ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualização ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Pré-processamento ─────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── Modelos de Classificação ──────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# ── Métricas de Avaliação ─────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    ConfusionMatrixDisplay
)

# ── Balanceamento de Classes ──────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE

# ── Configurações Gerais ──────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')  # Suprime avisos para deixar o output limpo

# Semente aleatória para reproducibilidade dos resultados
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('✅ Bibliotecas importadas com sucesso!')

---
## 1️⃣ Compreensão do Problema e Carregamento dos Dados

O dataset contém **1.143 amostras de vinho** com 11 variáveis físico-químicas e uma nota de qualidade atribuída por especialistas (escala de 0 a 10).

Nossa **variável-alvo** será transformada em binária:
- **1 → Alta Qualidade**: nota ≥ 7
- **0 → Baixa/Média Qualidade**: nota < 7

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CARREGAMENTO DOS DADOS
# ─────────────────────────────────────────────────────────────────────────────
# Se estiver no Google Colab, faça o upload do arquivo WineQT.csv
# usando o painel de arquivos à esquerda (ícone de pasta), ou use o código
# abaixo para fazer o upload diretamente pelo navegador:

# from google.colab import files
# uploaded = files.upload()   # Selecione o arquivo WineQT.csv

# Após o upload (ou se já tiver o arquivo no ambiente), carregue assim:
df = pd.read_csv('WineQT.csv')

# Removendo coluna de ID — não é uma variável preditora
df = df.drop(columns=['Id'], errors='ignore')

print(f'📊 Dataset carregado: {df.shape[0]} linhas × {df.shape[1]} colunas')
df.head()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CRIAÇÃO DA VARIÁVEL-ALVO BINÁRIA
# ─────────────────────────────────────────────────────────────────────────────
# Transformamos a nota contínua em uma classificação binária conforme
# o enunciado do desafio:
#   1 = Alta Qualidade  (quality >= 7)
#   0 = Baixa/Média     (quality <  7)

df['high_quality'] = (df['quality'] >= 7).astype(int)

# Vamos verificar a distribuição da variável original e da binária criada
print('Distribuição das notas originais de qualidade:')
print(df['quality'].value_counts().sort_index())
print()
print('Distribuição da variável-alvo binária (high_quality):')
print(df['high_quality'].value_counts())
print()

# Percentual de vinhos por classe
pct = df['high_quality'].value_counts(normalize=True) * 100
print(f'  Baixa/Média qualidade (0): {pct[0]:.1f}%')
print(f'  Alta qualidade        (1): {pct[1]:.1f}%')

---
## 2️⃣ Análise Exploratória de Dados (EDA)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VISÃO GERAL DO DATASET
# ─────────────────────────────────────────────────────────────────────────────
print('=' * 60)
print('INFORMAÇÕES GERAIS DO DATASET')
print('=' * 60)
df.info()
print()
print('Valores ausentes por coluna:')
print(df.isnull().sum())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ESTATÍSTICAS DESCRITIVAS
# Média, desvio padrão, mín, máx e quartis de cada variável
# ─────────────────────────────────────────────────────────────────────────────
df.describe().round(3)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# GRÁFICO: BALANCEAMENTO DAS CLASSES
# É importante verificar se as classes estão equilibradas antes de treinar.
# Se houver muito desequilíbrio, os modelos podem ficar viciados na classe
# majoritária, ignorando a minoritária.
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Subplot 1: Distribuição das notas originais ---
counts = df['quality'].value_counts().sort_index()
sns.barplot(x=counts.index, y=counts.values, ax=axes[0], palette='viridis')
axes[0].set_title('Distribuição das Notas Originais de Qualidade', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Nota de Qualidade')
axes[0].set_ylabel('Quantidade de Amostras')
for bar, val in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', va='bottom', fontsize=10)

# --- Subplot 2: Balanceamento da variável binária ---
bin_counts = df['high_quality'].value_counts()
labels = ['Baixa/Média\n(nota < 7)', 'Alta\n(nota ≥ 7)']
colors = ['#E07B54', '#5B9BD5']
axes[1].pie(bin_counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Balanceamento da Variável-Alvo Binária', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('class_balance.png', bbox_inches='tight')
plt.show()
print('\n💡 Observação: O dataset é desbalanceado (~86% vs ~14%).')
print('   Isso será tratado na etapa de pré-processamento com SMOTE.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# GRÁFICO: DISTRIBUIÇÃO DE CADA VARIÁVEL
# Histogramas com curva KDE (estimativa de densidade) para entender
# o formato da distribuição de cada feature.
# ─────────────────────────────────────────────────────────────────────────────
features = [c for c in df.columns if c not in ['quality', 'high_quality']]

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(features):
    # hue='high_quality' mostra como a distribuição difere entre as classes
    sns.histplot(data=df, x=col, hue='high_quality', kde=True, ax=axes[i],
                 palette={0: '#E07B54', 1: '#5B9BD5'}, alpha=0.6, bins=30)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].legend(title='Qualidade', labels=['Alta (1)', 'Baixa/Média (0)'], fontsize=8)

# Remove subplots extras
for j in range(len(features), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Distribuição das Variáveis por Classe de Qualidade', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('distributions.png', bbox_inches='tight')
plt.show()

print('\n💡 Observe como alcohol e sulphates têm distribuições mais separadas')
print('   entre as classes — isso indica boa capacidade preditiva!')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# GRÁFICO: BOXPLOTS POR CLASSE
# Boxplots mostram a mediana, quartis e outliers de cada variável,
# separados por classe. Ótimo para identificar outliers e diferenças
# entre grupos.
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.boxplot(data=df, x='high_quality', y=col, ax=axes[i],
                palette={0: '#E07B54', 1: '#5B9BD5'})
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Qualidade (0=Baixa, 1=Alta)')

for j in range(len(features), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Boxplots das Variáveis por Classe de Qualidade', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('boxplots.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DETECÇÃO DE OUTLIERS COM IQR (Intervalo Interquartil)
# Método: valores abaixo de Q1 - 1.5*IQR ou acima de Q3 + 1.5*IQR
# são considerados outliers.
# ─────────────────────────────────────────────────────────────────────────────
print('Quantidade de outliers por variável (método IQR):\n')
outlier_summary = {}
for col in features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    pct_out = n_out / len(df) * 100
    outlier_summary[col] = n_out
    print(f'  {col:<28}: {n_out:>4} outliers ({pct_out:.1f}%)')

print()
print('💡 Optamos por MANTER os outliers, pois em dados de vinho eles podem')
print('   representar características reais de amostras extremas, não erros.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MAPA DE CORRELAÇÃO (Heatmap)
# A correlação de Pearson mede a relação linear entre as variáveis.
# Valores próximos a +1 ou -1 indicam forte correlação.
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 10))

# Calcula a matriz de correlação (excluindo 'quality' original)
corr_cols = features + ['high_quality']
corr_matrix = df[corr_cols].corr()

# Máscara para ocultar o triângulo superior (evita repetição)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 9})

ax.set_title('Mapa de Correlação entre Variáveis', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

# Destacar correlações com a variável-alvo
print('\nCorrelação de cada variável com a qualidade binária (high_quality):')
target_corr = corr_matrix['high_quality'].drop('high_quality').sort_values(key=abs, ascending=False)
for var, val in target_corr.items():
    direction = '▲' if val > 0 else '▼'
    print(f'  {direction} {var:<28}: {val:+.3f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# INTERPRETAÇÃO DAS CORRELAÇÕES MAIS RELEVANTES
# ─────────────────────────────────────────────────────────────────────────────
print('=' * 65)
print('INSIGHTS DAS CORRELAÇÕES COM A QUALIDADE')
print('=' * 65)
insights = {
    'alcohol':            'Correlação POSITIVA forte. Vinhos com mais álcool tendem\n'
                          '                     a ter notas mais altas. Provável indicador de uva bem madura.',
    'volatile acidity':   'Correlação NEGATIVA forte. Alta acidez volátil (acético) causa\n'
                          '                     sabor de vinagre, prejudicando a qualidade.',
    'sulphates':          'Correlação POSITIVA moderada. Sulfatos são conservantes e\n'
                          '                     antimicrobianos; contribuem positivamente em níveis moderados.',
    'citric acid':        'Correlação POSITIVA moderada. Ácido cítrico adiciona frescor\n'
                          '                     e equilíbrio ao sabor do vinho.',
    'density':            'Correlação NEGATIVA moderada. Maior densidade geralmente indica\n'
                          '                     mais açúcar e menos álcool, associado a qualidade menor.',
    'chlorides':          'Correlação NEGATIVA fraca. Alto teor de cloretos (sal) pode\n'
                          '                     impactar negativamente o sabor.',
    'residual sugar':     'Correlação próxima de zero. O açúcar residual isoladamente\n'
                          '                     não é bom preditor da qualidade neste dataset.',
}
for var, desc in insights.items():
    print(f'\n  📌 {var}:')
    print(f'     {desc}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PAIRPLOT DAS PRINCIPAIS VARIÁVEIS
# Visualiza a relação entre as variáveis mais correlacionadas com a
# qualidade, coloridas pela classe-alvo.
# ─────────────────────────────────────────────────────────────────────────────
top_features = ['alcohol', 'volatile acidity', 'sulphates', 'citric acid', 'high_quality']

g = sns.pairplot(df[top_features], hue='high_quality', diag_kind='kde',
                 palette={0: '#E07B54', 1: '#5B9BD5'}, plot_kws={'alpha': 0.4})
g.fig.suptitle('Pairplot — Top 4 Variáveis vs Qualidade', y=1.02, fontsize=14, fontweight='bold')
g._legend.set_title('Qualidade')
for text, label in zip(g._legend.texts, ['Baixa/Média (0)', 'Alta (1)']):
    text.set_text(label)
plt.savefig('pairplot.png', bbox_inches='tight')
plt.show()

---
## 3️⃣ Pré-processamento dos Dados

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TRATAMENTO DE DADOS FALTANTES
# ─────────────────────────────────────────────────────────────────────────────
missing = df.isnull().sum()
print('Valores ausentes por coluna:')
print(missing[missing > 0] if missing.sum() > 0 else '  ✅ Nenhum valor ausente encontrado!')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE ENGINEERING
# Criamos novas features combinando variáveis existentes que podem
# capturar relações não-lineares relevantes para a qualidade.
# ─────────────────────────────────────────────────────────────────────────────

# Ratio álcool / acidez total — captura o equilíbrio entre força alcoólica
# e acidez, que é fundamental para a percepção de qualidade em vinhos
df['alcohol_acidity_ratio'] = df['alcohol'] / (df['fixed acidity'] + df['volatile acidity'] + 1e-6)

# Ratio sulfatos / cloretos — mede o equilíbrio entre conservantes (bons)
# e sal (ruim) no vinho
df['sulphate_chloride_ratio'] = df['sulphates'] / (df['chlorides'] + 1e-6)

# Total de acidez — soma das três componentes ácidas
df['total_acidity'] = df['fixed acidity'] + df['volatile acidity'] + df['citric acid']

# Razão SO2 livre / total — indica eficiência da proteção por dióxido de enxofre
df['so2_ratio'] = df['free sulfur dioxide'] / (df['total sulfur dioxide'] + 1e-6)

print('✅ Novas features criadas:')
print('  • alcohol_acidity_ratio  — equilíbrio álcool vs acidez')
print('  • sulphate_chloride_ratio — balanço conservante vs sal')
print('  • total_acidity          — acidez total combinada')
print('  • so2_ratio              — eficiência do SO2 livre')
print(f'\nTotal de features agora: {df.shape[1] - 2} (excluindo quality e high_quality)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SEPARAÇÃO EM FEATURES (X) E ALVO (y)
# ─────────────────────────────────────────────────────────────────────────────
# Removemos as duas colunas de qualidade: usamos apenas 'high_quality' como alvo
X = df.drop(columns=['quality', 'high_quality'])
y = df['high_quality']

print(f'Features (X): {X.shape[1]} colunas, {X.shape[0]} linhas')
print(f'Alvo     (y): {y.shape[0]} linhas')
print(f'\nColunas usadas como features:')
for col in X.columns:
    print(f'  • {col}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DIVISÃO TREINO / TESTE
# Usamos stratify=y para garantir que a proporção das classes seja
# mantida tanto no conjunto de treino quanto no de teste.
# ─────────────────────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,      # 20% para teste, 80% para treino
    random_state=RANDOM_STATE,
    stratify=y           # Mantém proporção das classes
)

print(f'Tamanho do conjunto de treino: {X_train.shape[0]} amostras')
print(f'Tamanho do conjunto de teste:  {X_test.shape[0]} amostras')
print()
print('Proporção das classes no treino:')
print(y_train.value_counts(normalize=True).round(3))
print()
print('Proporção das classes no teste:')
print(y_test.value_counts(normalize=True).round(3))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BALANCEAMENTO DAS CLASSES COM SMOTE
# SMOTE (Synthetic Minority Over-sampling Technique) cria amostras
# SINTÉTICAS da classe minoritária (vinhos de alta qualidade) para
# equilibrar o dataset de treino.
#
# IMPORTANTE: SMOTE é aplicado APENAS no conjunto de treino para
# evitar vazamento de informação para o teste.
# ─────────────────────────────────────────────────────────────────────────────
smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('Antes do SMOTE (treino):')
print(f'  Classe 0: {(y_train == 0).sum()} amostras')
print(f'  Classe 1: {(y_train == 1).sum()} amostras')
print()
print('Depois do SMOTE (treino):')
print(f'  Classe 0: {(y_train_bal == 0).sum()} amostras')
print(f'  Classe 1: {(y_train_bal == 1).sum()} amostras')
print()
print('✅ Classes balanceadas! O conjunto de treino agora tem proporção igual.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# NORMALIZAÇÃO (STANDARDSCALER)
# StandardScaler transforma cada variável para ter média 0 e desvio
# padrão 1. Isso é essencial para a Regressão Logística, que é sensível
# à escala das features.
# O Random Forest NÃO precisa de normalização, mas aplicamos para
# padronizar o pipeline.
# ─────────────────────────────────────────────────────────────────────────────
scaler = StandardScaler()

# Fit APENAS no treino — depois aplicamos a mesma transformação no teste
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled  = scaler.transform(X_test)  # Usa parâmetros do treino!

print('✅ Normalização aplicada:')
print(f'  Média pós-escala (deve ser ≈ 0): {X_train_scaled.mean():.6f}')
print(f'  Desvio padrão   (deve ser ≈ 1): {X_train_scaled.std():.6f}')

---
## 4️⃣ Desenvolvimento dos Modelos

Treinaremos **três modelos de classificação** para comparar:

| Modelo | Tipo | Complexidade |
|---|---|---|
| **Regressão Logística** | Linear | Simples, interpretável |
| **Random Forest** | Ensemble (bagging) | Moderada, robusto |
| **Gradient Boosting** | Ensemble (boosting) | Alta, muito poderoso |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TREINAMENTO DOS MODELOS
# ─────────────────────────────────────────────────────────────────────────────

# 1. REGRESSÃO LOGÍSTICA
# Modelo linear clássico para classificação binária. Estima a probabilidade
# de uma amostra pertencer à classe 1 via função sigmoide.
# max_iter=1000 para garantir convergência com os dados normalizados.
lr = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    C=1.0  # Regularização padrão (L2)
)
lr.fit(X_train_scaled, y_train_bal)
print('✅ Regressão Logística treinada')

# 2. RANDOM FOREST
# Conjunto de múltiplas árvores de decisão (100 árvores) que votam
# na classificação final. Muito robusto a outliers e não precisa de
# normalização, mas aplicamos para manter consistência no pipeline.
rf = RandomForestClassifier(
    n_estimators=200,    # 200 árvores
    max_depth=10,        # Profundidade máxima para evitar overfitting
    min_samples_split=5,
    random_state=RANDOM_STATE,
    n_jobs=-1            # Usa todos os núcleos do processador
)
rf.fit(X_train_scaled, y_train_bal)
print('✅ Random Forest treinado')

# 3. GRADIENT BOOSTING
# Constrói árvores sequencialmente, onde cada nova árvore corrige os
# erros da anterior. Geralmente o modelo mais poderoso entre os três.
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,  # Taxa de aprendizado
    max_depth=4,
    random_state=RANDOM_STATE
)
gb.fit(X_train_scaled, y_train_bal)
print('✅ Gradient Boosting treinado')

print('\n🎯 Todos os modelos foram treinados com sucesso!')

---
## 5️⃣ Avaliação e Comparação dos Modelos

Usaremos as seguintes métricas:

- **Accuracy**: % de acertos totais — mas enganosa em dados desbalanceados
- **Precision**: dos que previu como Alta Qualidade, quantos realmente são?
- **Recall**: dos vinhos de Alta Qualidade reais, quantos foram identificados?
- **F1-Score**: média harmônica entre Precision e Recall — equilíbrio entre os dois
- **ROC-AUC**: capacidade de distinguir as classes (1.0 = perfeito, 0.5 = aleatório)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FUNÇÃO DE AVALIAÇÃO
# Centraliza o cálculo de todas as métricas para facilitar a comparação
# entre modelos.
# ─────────────────────────────────────────────────────────────────────────────
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """
    Avalia um modelo de classificação e retorna um dicionário com métricas.
    
    Parâmetros:
        model: modelo já treinado
        X_train, y_train: dados de treino
        X_test, y_test: dados de teste
        model_name: nome do modelo (string)
    """
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]  # Probabilidade da classe 1

    # Validação cruzada com 5 folds estratificados no conjunto de treino
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)

    metrics = {
        'Modelo':        model_name,
        'Accuracy':      accuracy_score(y_test, y_pred),
        'Precision':     precision_score(y_test, y_pred),
        'Recall':        recall_score(y_test, y_pred),
        'F1-Score':      f1_score(y_test, y_pred),
        'ROC-AUC':       roc_auc_score(y_test, y_proba),
        'F1 CV (média)': cv_scores.mean(),
        'F1 CV (std)':   cv_scores.std(),
    }
    return metrics

# Avalia os três modelos
results = []
for model, name in [(lr, 'Regressão Logística'), (rf, 'Random Forest'), (gb, 'Gradient Boosting')]:
    metrics = evaluate_model(model, X_train_scaled, y_train_bal, X_test_scaled, y_test, name)
    results.append(metrics)
    print(f'✅ {name} avaliado')

# Cria DataFrame de resultados
results_df = pd.DataFrame(results).set_index('Modelo')
print('\n')
print('=' * 70)
print('TABELA COMPARATIVA DE DESEMPENHO DOS MODELOS')
print('=' * 70)
print(results_df.round(4).to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MATRIZES DE CONFUSÃO
# Mostra para cada modelo:
#   - Verdadeiros Positivos (VP): Alta Qualidade prevista e real
#   - Falsos Positivos   (FP): previu Alta Qualidade, mas era Baixa
#   - Verdadeiros Negativos (VN): Baixa Qualidade prevista e real
#   - Falsos Negativos   (FN): previu Baixa, mas era Alta Qualidade
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
models_list = [(lr, 'Regressão Logística'), (rf, 'Random Forest'), (gb, 'Gradient Boosting')]

for ax, (model, name) in zip(axes, models_list):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Baixa/Média\n(0)', 'Alta\n(1)'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=12, fontweight='bold')

plt.suptitle('Matrizes de Confusão — Conjunto de Teste', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CURVAS ROC
# A curva ROC plota a Taxa de Verdadeiros Positivos vs a Taxa de Falsos
# Positivos para diferentes limiares de decisão. A área sob a curva
# (AUC) resume o desempenho: 1.0 = perfeito, 0.5 = aleatório.
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['#E07B54', '#5B9BD5', '#70AD47']
for (model, name), color in zip(models_list, colors):
    RocCurveDisplay.from_estimator(model, X_test_scaled, y_test,
                                   ax=ax, name=name, color=color)

# Linha diagonal = classificador aleatório (AUC = 0.5)
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', alpha=0.6, label='Aleatório (AUC = 0.5)')

ax.set_title('Curvas ROC — Comparação dos Modelos', fontsize=14, fontweight='bold')
ax.set_xlabel('Taxa de Falsos Positivos')
ax.set_ylabel('Taxa de Verdadeiros Positivos')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# GRÁFICO COMPARATIVO DE MÉTRICAS
# ─────────────────────────────────────────────────────────────────────────────
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
plot_df = results_df[metrics_to_plot]

x = np.arange(len(metrics_to_plot))
width = 0.25
fig, ax = plt.subplots(figsize=(13, 6))

palette = ['#E07B54', '#5B9BD5', '#70AD47']
for i, (model_name, row) in enumerate(plot_df.iterrows()):
    bars = ax.bar(x + i * width, row.values, width, label=model_name,
                  color=palette[i], edgecolor='white', alpha=0.9)
    # Adiciona valor em cima de cada barra
    for bar, val in zip(bars, row.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Comparação de Desempenho entre Modelos', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('metrics_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RELATÓRIO DETALHADO DO MELHOR MODELO
# ─────────────────────────────────────────────────────────────────────────────

# Identifica o melhor modelo por F1-Score
best_model_name = results_df['F1-Score'].idxmax()
print(f'🏆 Melhor modelo por F1-Score: {best_model_name}')
print()

# Busca o objeto do modelo pelo nome
model_map = {'Regressão Logística': lr, 'Random Forest': rf, 'Gradient Boosting': gb}
best_model = model_map[best_model_name]
y_pred_best = best_model.predict(X_test_scaled)

print('Classification Report completo (melhor modelo):')
print(classification_report(y_test, y_pred_best,
                             target_names=['Baixa/Média (0)', 'Alta Qualidade (1)']))

---
## 6️⃣ Interpretação dos Resultados

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# IMPORTÂNCIA DAS FEATURES — RANDOM FOREST
# O Random Forest calcula naturalmente a importância de cada variável
# com base na redução de impureza (Gini) ao longo de todas as árvores.
# ─────────────────────────────────────────────────────────────────────────────
feature_names = X.columns.tolist()
rf_importances = pd.Series(rf.feature_importances_, index=feature_names)
rf_importances = rf_importances.sort_values(ascending=True)  # Ordenado para gráfico horizontal

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#5B9BD5' if i >= len(feature_names) - 5 else '#B0C4DE'
          for i in range(len(rf_importances))]
bars = ax.barh(rf_importances.index, rf_importances.values, color=colors, edgecolor='white')

for bar, val in zip(bars, rf_importances.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_title('Importância das Features — Random Forest\n(Top 5 em azul escuro)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importância relativa (soma = 1.0)')
plt.tight_layout()
plt.savefig('feature_importance_rf.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# COEFICIENTES DA REGRESSÃO LOGÍSTICA
# Os coeficientes indicam o peso de cada variável na decisão do modelo.
# Positivo = aumentar a variável aumenta a chance de Alta Qualidade.
# Negativo = aumentar a variável reduz a chance de Alta Qualidade.
# ─────────────────────────────────────────────────────────────────────────────
lr_coefs = pd.Series(lr.coef_[0], index=feature_names).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#E07B54' if v < 0 else '#5B9BD5' for v in lr_coefs.values]
bars = ax.barh(lr_coefs.index, lr_coefs.values, color=colors, edgecolor='white')

ax.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Coeficientes da Regressão Logística\n(Azul = favorece Alta Qualidade | Vermelho = desfavorece)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Coeficiente (dados normalizados)')
plt.tight_layout()
plt.savefig('lr_coefficients.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RESUMO EXECUTIVO DOS RESULTADOS
# ─────────────────────────────────────────────────────────────────────────────
print('=' * 70)
print('RESUMO EXECUTIVO — TECH CHALLENGE FIAP')
print('Classificação de Qualidade de Vinhos com Machine Learning')
print('=' * 70)

print()
print('── COMPARATIVO FINAL ─────────────────────────────────────────────────')
print(results_df[['Accuracy', 'F1-Score', 'ROC-AUC']].round(4).to_string())

best_row = results_df.loc[best_model_name]
print()
print(f'── MODELO VENCEDOR: {best_model_name} ────────────────────────────────')
print(f'   F1-Score : {best_row["F1-Score"]:.2%}')
print(f'   ROC-AUC  : {best_row["ROC-AUC"]:.2%}')
print(f'   Accuracy : {best_row["Accuracy"]:.2%}')

print()
print('── TOP 5 VARIÁVEIS MAIS IMPORTANTES (Random Forest) ─────────────────')
top5 = rf_importances.sort_values(ascending=False).head(5)
for i, (feat, imp) in enumerate(top5.items(), 1):
    print(f'   {i}. {feat:<30}: {imp:.3f}')

print()
print('── IMPLICAÇÕES PARA A PRODUÇÃO ───────────────────────────────────────')
print()
print('   1. ÁLCOOL: A variável mais preditiva. Produtores devem focar em')
print('      processos fermentativos que maximizem o teor alcoólico.')
print()
print('   2. ACIDEZ VOLÁTIL: Deve ser monitorada e mantida baixa, pois')
print('      está fortemente associada a vinhos de baixa qualidade.')
print()
print('   3. SULFATOS: Dentro dos limites seguros, contribuem positivamente.')
print('      Ajustes na dosagem podem elevar a nota percebida do vinho.')
print()
print('   4. O modelo pode ser integrado ao controle de qualidade para')
print('      triar lotes em tempo real, priorizando análise sensorial')
print('      apenas nos casos classificados na fronteira de decisão.')
print()
print('=' * 70)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SALVAR RESULTADOS
# Exporta a tabela de métricas para CSV e os gráficos já foram salvos
# como PNG nas células anteriores.
# ─────────────────────────────────────────────────────────────────────────────
results_df.round(4).to_csv('model_metrics.csv')
print('📁 Arquivos salvos no diretório atual:')
print('   • model_metrics.csv         — tabela comparativa de métricas')
print('   • class_balance.png         — balanceamento das classes')
print('   • distributions.png         — histogramas por classe')
print('   • boxplots.png              — boxplots por classe')
print('   • correlation_heatmap.png   — mapa de correlação')
print('   • pairplot.png              — pairplot variáveis principais')
print('   • confusion_matrices.png    — matrizes de confusão')
print('   • roc_curves.png            — curvas ROC comparativas')
print('   • metrics_comparison.png    — comparativo de métricas')
print('   • feature_importance_rf.png — importância das features (RF)')
print('   • lr_coefficients.png       — coeficientes da reg. logística')
print()
print('✅ Pipeline completo! Bom Tech Challenge!')